# FFM-OSLO v2.1 — 单库先跑通，加库时从零重训

**架构只有 FFM + FiLM，没有别的。** 相对 v2.0 的改动全是"修 bug"或"把接口接上"，
不是新方法：

| # | 改动 | 为什么 |
|---|---|---|
| 1 | `DATASETS` 成为唯一真相源 | v2.0 有两处各自为政的路径（Cell 2b 的 `DATA_DIRS` 只做检查，Cell 3 的 `left_dir` 才训练）。往前者加库，检查会过、训练集不变 |
| 2 | FiLM 跨库中心化 | v2.0 说 `num_datasets=1` 是恒等映射——初始化时是，训练后不是。中心化后**单库时严格恒等**，这一版的 CARLab 结果就是一个干净的纯 FFM 基线 |
| 3 | 编码器侧共享参数的逆 FiLM | **零新增参数**，单库时完全不起作用。多库时才有意义：v2.0 的 FiLM 只挂解码器输出，encoder 看的还是带染色的 `x`，染色照样进 `z`。不想要就把 `tie_encoder_film` 设 `False`，行为等同 v2.0 |
| 4 | train / val / test 三分，划分冻结到磁盘 | v2.0 只有 train/val，val 同时当 test 用 |
| 5 | `SpectralGradientLoss` 加频段掩码 | v2.0 对**全部** bin 求一阶差分，没有掩码——低频那几档的差分会混进损失 |
| 6 | bin 0–1 载入时压平 | bin 0 是 DC：3.3% 的像素低于 −30 dB、最低 −80 dB，方差是 bin 20 的 1.41 倍。只做损失掩码挡不住它进 encoder |

**加库的做法：把 `DATASETS` 里的注释取消 → `num_datasets` 自动变 → 从零重训。**
不做 warm-start 续训——那样得到的数字和单库阶段不可比（训练预算和见过的数据都不同），
论证"并库有收益"本来就需要两次从零训练。既然如此，`expand_film_rows` /
`rebuild_optimizer_state` / 冻结主干 / 分组学习率一样都不需要，这一版全部删掉了。

**频段**：hard cap 16 kHz（`hrirfilt` 在 16.31 kHz 被数字滤波器切断，见
`analysis/band_limit_audit.py`），损失下限 500 Hz。129 → **93 bins**。

In [ ]:
# ==========================================================================
# Cell 0 — 数据准备（由 DATASETS 驱动）
# ==========================================================================
import os, zipfile
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
EXTRACT_DIR = '/content/dataset/'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# 每个库一条。id 决定 FiLM 的行号；加库就把下面的注释打开，然后【从零重训】。
DATASETS = [
    dict(id=0, tag='CARLab',
         zips=[f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_left.zip',
               f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_right.zip'],
         left=f'{EXTRACT_DIR}CARLab_44k1_left',
         right=f'{EXTRACT_DIR}CARLab_44k1_right',
         suffix=('_left_full.npy', '_right_full.npy'),
         split=(0.80, 0.10, 0.10)),        # train / val / test
    # dict(id=1, tag='HUTUBS',
    #      zips=[f'{DRIVE}/ELEC5021/HUTUBS/HUTUBS_44k1_left.zip',
    #            f'{DRIVE}/ELEC5021/HUTUBS/HUTUBS_44k1_right.zip'],
    #      left=f'{EXTRACT_DIR}HUTUBS_44k1_left',
    #      right=f'{EXTRACT_DIR}HUTUBS_44k1_right',
    #      suffix=('_left_full.npy', '_right_full.npy'),
    #      split=(0.60, 0.15, 0.25)),      # 自带 58 个 head mesh -> 主力 test 集
    # dict(id=2, tag='SONICOM',
    #      zips=[f'{DRIVE}/ELEC5021/SONICOM/SONICOM_44k1_left.zip',
    #            f'{DRIVE}/ELEC5021/SONICOM/SONICOM_44k1_right.zip'],
    #      left=f'{EXTRACT_DIR}SONICOM_44k1_left',
    #      right=f'{EXTRACT_DIR}SONICOM_44k1_right',
    #      suffix=('_left_full.npy', '_right_full.npy'),
    #      split=(0.75, 0.10, 0.15)),
]
DATASETS = sorted(DATASETS, key=lambda d: d['id'])
assert [d['id'] for d in DATASETS] == list(range(len(DATASETS))), 'id 必须从 0 连续'

for d in DATASETS:
    for zp in d['zips']:
        if not os.path.exists(zp):
            print(f'[缺失] {zp}'); continue
        with zipfile.ZipFile(zp) as z:
            z.extractall(EXTRACT_DIR)
    for side in ('left', 'right'):
        p = d[side]
        n = len([f for f in os.listdir(p) if f.endswith('.npy')]) if os.path.isdir(p) else 0
        print(f"{d['tag']:<9} {side:<6} {p}  ({n} npy)")

MIRROR_SRC = f'{DRIVE}/healpix_mirror_idx.npy'
print('\nmirror idx:', '找到' if os.path.exists(MIRROR_SRC) else '★缺失，Cell 3 会现场生成')

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 1 — 采样率 / 频率轴一致性检查（并库前必跑）
#
# 同一个 bin 索引在不同采样率下对应【不同的物理频率】：
#   48   kHz / 256 : df = 187.50 Hz,  bin 64 = 12000 Hz
#   44.1 kHz / 256 : df = 172.27 Hz,  bin 64 = 11025 Hz
# 在 notch 带错位接近 1 kHz —— 和一个 notch 本身的宽度同量级。
# 混着训不会报错，只会悄悄变差。
# ==========================================================================
import os, json
import numpy as np

EXPECT_FS, EXPECT_NFFT, EXPECT_BINS, EXPECT_NPIX = 44100, 256, 129, 3072
print(f'目标: fs={EXPECT_FS} Hz, nfft={EXPECT_NFFT}, {EXPECT_BINS} bins, '
      f'df={EXPECT_FS/EXPECT_NFFT:.3f} Hz, fmax={EXPECT_FS/2:.0f} Hz')
print('=' * 82)

ok = True
for d in DATASETS:
    for side in ('left', 'right'):
        p = d[side]
        if not os.path.isdir(p):
            print(f"  {d['tag']+'-'+side:<16} ★ 目录不存在: {p}"); ok = False; continue
        mp = os.path.join(p, 'meta.json')
        m = json.load(open(mp)) if os.path.exists(mp) else {}
        fs, nfft, nb = m.get('sample_rate'), m.get('nfft'), m.get('n_bins')
        files = sorted(f for f in os.listdir(p) if f.endswith('.npy'))
        x = np.load(os.path.join(p, files[0]))
        bad = []
        if fs != EXPECT_FS:     bad.append(f'fs={fs}')
        if nfft != EXPECT_NFFT: bad.append(f'nfft={nfft}')
        if nb != EXPECT_BINS:   bad.append(f'n_bins={nb}')
        if x.shape != (EXPECT_BINS, EXPECT_NPIX): bad.append(f'shape={x.shape}')
        ok &= not bad
        print(f"  {d['tag']+'-'+side:<16}{len(files):>5} 文件 | "
              f"{'meta.json' if m else '★ 无 meta':<10} | fs={fs} nfft={nfft} | "
              f"shape={x.shape} | {'OK' if not bad else '★ ' + ', '.join(bad)}")

print('=' * 82)
print('全部一致 ✔' if ok else '★ 有不一致项，先修好再训练')
assert ok, '数据集配置不一致'

# 频率上限：CARLab 在 16.31 kHz 被切断（analysis/band_limit_audit.py 已证实那是
# 滤波器设计造成的，不是测量上限）。多库训练取【最小】上限。
FREQ_CAP_HZ = 16000
DF = EXPECT_FS / EXPECT_NFFT
N_BINS = int(np.floor(FREQ_CAP_HZ / DF)) + 1
print(f'\n频率上限 {FREQ_CAP_HZ} Hz -> 保留 bins 0..{N_BINS-1} ({N_BINS}/{EXPECT_BINS}), '
      f'实际 fmax {(N_BINS-1)*DF:.0f} Hz')

In [ ]:
# ==========================================================================
# Cell 2 — 模型定义（GELU + FiLM）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- 球面卷积
class OSLOConv(nn.Module):
    """HEALPix 上的各向异性卷积：每个像素 + 8 邻居，9 个独立权重。"""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """频率隔离的残差块，处理 (B*F, C, N)。
    ★ v2: ReLU -> GELU。HRTF 的 log-magnitude 是双极量（负值表示衰减），
      ReLU 会把负半轴直接截断；GELU 平滑且保留负值，也与 fc 层一致。"""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """为每个数据集学一条【逐频率】的 scale/shift 曲线。

    ★★ 约束一：只按频率，【不】按方向。
      per-frequency 仿射 = 一条 EQ 曲线（93 个自由度/数据集）。它能吸收的只有
      "整个数据集共有的、与方向无关的染色" —— 扬声器/麦克风响应、均衡约定、
      整体电平。它在结构上【无法】表达方向相关的差异，因此人群解剖差异
      （不同库的耳朵大小分布不同）仍然必须留在 z 里。
      对比：按 (频率 x 方向) 减去每库均值 = 3072x93 个自由度，足以把真实的
      人群结构一并扣掉 —— 那正是 Craig 指出的 wrong direction。
      两者表达能力差 3072 倍，不是同一件事的两种写法。

    ★★ 约束二：跨数据集中心化（v2.1）。
      v2.0 的注释说 num_datasets=1 时是恒等映射 —— 初始化时是，训练之后不是：
      log_gamma/beta 可训练，会学出一条本库专属的全局 EQ，和网络本体的输出增益
      互相抵消，是个多余的自由度。
      减掉跨库均值之后，FiLM 只表达【库与库之间的相对差异】，绝对部分留给网络
      本体。于是 num_datasets=1 时【严格恒等】—— 单库这一版就是干净的纯 FFM。
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- 编码器 / 解码器

class Isolated_OSLO_Encoder(nn.Module):
    """每个频点独立编码 -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM 频率条件化解码器 + 输出端 per-dataset FiLM"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """★ v2.1: 编码器输入端加一次【共享参数的逆 FiLM】—— 零新增参数。

    v2.0 是   z = encoder(x);  out = film(decoder(z))
    问题：FiLM 存在的理由是吸收各库的测量染色，但染色留在 x 里，encoder 照单
    全收，于是染色照样进 z —— 而 z 正是下游 geometry→latent 回归要用的东西。
    输出端的 FiLM 只让重建损失对得上，没把 z 清干净。

    v2.1 是   x_norm = (x - beta)/gamma        <- 去染色
              z      = encoder(x_norm)
              out    = gamma * decoder(z) + beta   <- 回到原始空间，损失照旧算 raw

    用的是【同一组】FiLM 参数，不是第二组。配合中心化，也正好消掉
    "gamma 整体放大、网络整体缩小" 这个平凡的退化方向。
    单库时 FiLM 恒等，所以这一句【完全不起作用】。
    不想要就把 tie_encoder_film 设 False，行为等同 v2.0。
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('模型定义完成 (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 配置 · train/val/test 划分 · 损失
# ==========================================================================
import os, json, random, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
import healpy as hp

config = {
    'model_type': 'FFM-OSLO-v2.1',
    'nside': 16, 'latent_dim': 64, 'freq_embed_dim': 32,
    'sample_rate': EXPECT_FS, 'num_bins_raw': EXPECT_BINS,
    'freq_cap_hz': FREQ_CAP_HZ,
    'flatten_bins': 2,          # bin 0,1 载入时压平，见下方
    'band_lo_hz': 500,          # 损失下限
    'num_datasets': len(DATASETS),
    'tie_encoder_film': True,   # 单库时无影响；设 False = v2.0 行为
    'lr': 1e-4, 'batch_size': 8, 'weight_decay': 1e-4,
    'w_mse': 1.0, 'w_spectral': 0.1,
    'tau': 1.0,                 # 采样温度，见下方。单库时无意义
    'save_dir': '/content/drive/MyDrive/HRTF_Checkpoints',
}
config['num_bins'] = N_BINS
config['band_hi_hz'] = int(round((N_BINS - 1) * DF))

# --------------------------- ★ 低频的处理 ---------------------------
# 为什么下限是 500 Hz、以及为什么光靠损失掩码不够（CARLab 218 人实测）：
#
#   bin 0 (  0.0 Hz, DC) : 3.26% 的像素 < -30 dB, 最低 -80 dB,
#                          方差是 bin 20 的 1.41 倍  <- 离群点, 不是信号
#   bin 1 (172.3 Hz)     : 1.46% < -20 dB, 方差 0.60x
#                          跨方向 std 4.0 dB, 但 ka=0.28 时物理上应该 ~1 dB
#   bin 2 (344.5 Hz)     : 0.21% < -20 dB, 方差 0.20x, 跨方向 std 1.8 dB
#                          <- 这一档看起来是【真的】, 跨方向 std 随频率单调上升
#
# 也就是说 500 Hz 略微保守（bin 2 其实可用），真正的问题在 bin 0/1。
# 只做损失掩码不够：那两档仍会经 encoder 进 z。所以载入时用 bin 2 的值把它们
# 压平 —— 去掉离群点，同时保持 93 的张量形状（硬删会让旧 checkpoint 失效）。
FLAT = config['flatten_bins']

resume = False              # Colab 断线后续训用；不改 DATASETS 时才可以开
fresh_epochs = 150
additional_epochs = 50

os.makedirs(config['save_dir'], exist_ok=True)
latest_path = os.path.join(config['save_dir'], 'FFM21_latest.pth')
best_path   = os.path.join(config['save_dir'], 'FFM21_best.pth')
split_path  = os.path.join(config['save_dir'], 'split_v21.json')

seed = 4
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

# ------------------------------ 镜像置换 ------------------------------
if os.path.exists(MIRROR_SRC):
    MIRROR = np.load(MIRROR_SRC)
else:
    npix = hp.nside2npix(config['nside'])
    th, ph = hp.pix2ang(config['nside'], np.arange(npix))
    MIRROR = hp.ang2pix(config['nside'], th, (-ph) % (2 * np.pi))
assert len(np.unique(MIRROR)) == len(MIRROR), '镜像不是双射'
assert np.array_equal(MIRROR[MIRROR], np.arange(len(MIRROR))), '镜像不是对合'
print('镜像置换 OK (精确双射且对合)')


# ------------------------------ 单耳数据集 ------------------------------
class MonauralHRTFDataset(Dataset):
    """每个样本 = 一只耳朵。右耳沿中矢面镜像，使全部样本进入"左耳坐标系"。
    v1 的方差分解显示 71.7% 的 latent 方差在编码 side —— 一个本来就知道的比特。"""
    def __init__(self, d, subject_ids_, mirror):
        self.items, self.mirror, self.dataset_id = [], mirror, d['id']
        sl, sr = d['suffix']
        for sid in sorted(subject_ids_):
            for side, sfx, root in (('left', sl, d['left']), ('right', sr, d['right'])):
                p = os.path.join(root, f'{sid}{sfx}')
                if os.path.exists(p):
                    self.items.append((sid, side, p))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        sid, side, path = self.items[i]
        x = np.load(path).astype(np.float32)[:N_BINS]
        if FLAT:
            x[:FLAT] = x[FLAT]
        if side == 'right':
            x = x[:, self.mirror]
        return torch.from_numpy(x), torch.tensor(self.dataset_id), f'{sid}_{side}'


def subject_ids(d):
    sfx = d['suffix'][0]
    return sorted({f[:-len(sfx)] for f in os.listdir(d['left']) if f.endswith(sfx)})


# --------------------- train / val / test：按【受试者】划分 ---------------------
# 为什么按人不按样本：同一人左右耳镜像后相关 0.978，按样本随机切 = 数据泄漏。
# 为什么每库【各自】切：小库有实打实的概率整体落进某一边。
# 为什么每库用【自己的 rng】(seed + id)：这样以后加库，已有库的划分一个字都不会变。
# 为什么要冻结到磁盘：seed 只在"数据没动过"时可复现。某个库多几个人，同一个
#   seed 就会给出不同划分 —— test 集当场污染。所以存下来 + 存名单 hash 做校验。
def build_split(datasets, seed, path):
    sp = json.load(open(path)) if os.path.exists(path) else {}
    changed = False
    for d in datasets:
        ids = subject_ids(d)
        h = hashlib.md5(','.join(ids).encode()).hexdigest()[:12]
        if d['tag'] in sp:
            if sp[d['tag']]['hash'] != h:
                raise RuntimeError(
                    f"{d['tag']} 的受试者名单变了（存档 {sp[d['tag']]['hash']} vs 现在 {h}）。"
                    '重新划分会污染 test 集 —— 先确认这是不是预期的。')
            continue
        rng = np.random.default_rng(seed + d['id'])      # 每库独立
        perm = list(ids); rng.shuffle(perm)
        ftr, fva, _ = d['split']
        n = len(perm); n_tr = int(round(ftr * n)); n_va = int(round(fva * n))
        sp[d['tag']] = {'train': perm[:n_tr], 'val': perm[n_tr:n_tr + n_va],
                        'test': perm[n_tr + n_va:], 'n_total': n, 'hash': h}
        changed = True
        print(f"  ★ 新划分 {d['tag']}（已有库不受影响）")
    if changed:
        json.dump(sp, open(path, 'w'), indent=1)
        print(f'  划分已冻结到 {path}  —— 以后不要删掉重建')
    else:
        print(f'  沿用已冻结的划分 {path}')
    return sp


SPLIT = build_split(DATASETS, seed, split_path)

tr_parts, va_parts, te_parts, rows = [], [], [], []
for d in DATASETS:
    s = SPLIT[d['tag']]
    tr_parts.append(MonauralHRTFDataset(d, s['train'], MIRROR))
    va_parts.append(MonauralHRTFDataset(d, s['val'],   MIRROR))
    te_parts.append(MonauralHRTFDataset(d, s['test'],  MIRROR))
    rows.append((d['id'], d['tag'], s['n_total'], len(s['train']), len(s['val']),
                 len(s['test']), len(tr_parts[-1]), len(va_parts[-1]), len(te_parts[-1])))

train_dataset = ConcatDataset(tr_parts)
val_dataset   = ConcatDataset(va_parts)
test_dataset  = ConcatDataset(te_parts)

print(f"\n{'id':>3} {'库':<9}{'人数':>6}{'train':>7}{'val':>5}{'test':>6}"
      f"{'tr耳':>7}{'va耳':>6}{'te耳':>6}")
for r in rows:
    print(f'{r[0]:>3} {r[1]:<9}{r[2]:>6}{r[3]:>7}{r[4]:>5}{r[5]:>6}{r[6]:>7}{r[7]:>6}{r[8]:>6}')
print(f'    合计  train {len(train_dataset)} 耳 | val {len(val_dataset)} 耳 | '
      f'test {len(test_dataset)} 耳（test 在最后一个 cell 之前不要碰）')


# ------------------------------ DataLoader ------------------------------
# tau: P(库 d) ∝ n_d^tau，库内均匀。
#   tau = 1（默认）等价于把三个库拼起来直接 shuffle —— 最朴素的做法。
#   tau = 0.5 会把小库的占比抬上来。什么时候需要它：并库训练时如果某个库的
#            val RMSE 明显掉队、或它那行 FiLM 一直贴着 0 不动，再改。
#            按 CARLab 218 / HUTUBS 96 / SONICOM 200 估算，tau=1 时 HUTUBS
#            只占 15%，tau=0.5 时 23%。先按 1 跑，有问题再说。
g = torch.Generator(); g.manual_seed(seed)
if len(DATASETS) > 1 and config['tau'] != 1.0:
    n = np.array([len(p) for p in tr_parts], float)
    P = n ** config['tau']; P /= P.sum()
    w = np.concatenate([np.full(int(ni), Pi / ni) for ni, Pi in zip(n, P)])
    print(f"\n温度采样 tau={config['tau']}: " +
          '  '.join(f'{d["tag"]} {p:.1%}' for d, p in zip(DATASETS, P)))
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                              sampler=WeightedRandomSampler(
                                  torch.as_tensor(w, dtype=torch.double),
                                  num_samples=int(n.sum()), replacement=True))
else:
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                              shuffle=True, generator=g)
val_loader  = DataLoader(val_dataset,  batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)


# ------------------------------ 频段掩码 ------------------------------
freqs = np.arange(N_BINS) * DF
band = torch.tensor(freqs >= config['band_lo_hz'])
band_dev = band.to(device)
print(f"\n损失频段 {config['band_lo_hz']}-{config['band_hi_hz']} Hz -> "
      f"{int(band.sum())}/{len(band)} 个频点（掩掉 bin 0..{int((~band).sum())-1}）")


# ------------------------------ 损失 ------------------------------
class SpectralGradientLoss(nn.Module):
    """沿频率轴的一阶差分 MSE —— 鼓励模型贴合 notch 的陡峭形状。
    ★ v2.1 加了频段掩码：v2.0 对【全部】bin 求差分，低频那几档会混进来。"""
    def __init__(self, band):
        super().__init__()
        self.register_buffer('m', band[1:] & band[:-1])

    def forward(self, pred, target):
        dp = (pred[:, 1:, :] - pred[:, :-1, :])[:, self.m, :]
        dt = (target[:, 1:, :] - target[:, :-1, :])[:, self.m, :]
        return F.mse_loss(dp, dt)


# ------------------------------ 模型 ------------------------------
model = FFM_OSLO_Autoencoder(
    nside=config['nside'], latent_dim_per_freq=config['latent_dim'],
    freq_embed_dim=config['freq_embed_dim'], num_datasets=config['num_datasets'],
    num_bins=config['num_bins'], tie_encoder_film=config['tie_encoder_film']).to(device)

optimizer = optim.Adam(model.parameters(), lr=config['lr'],
                       weight_decay=config['weight_decay'])
criterion_mse, criterion_mae = nn.MSELoss(), nn.L1Loss()
criterion_spectral = SpectralGradientLoss(band).to(device)

model.eval()
with torch.no_grad():
    dmy = torch.randn(2, N_BINS, 3072).to(device)
    o, z = model(dmy, torch.zeros(2, dtype=torch.long, device=device))
    print(f'\n输入 {tuple(dmy.shape)} | 输出 {tuple(o.shape)} | latent {tuple(z.shape)}')
    assert o.shape == dmy.shape

# 单库自检：FiLM 应当【严格】恒等
if config['num_datasets'] == 1:
    with torch.no_grad():
        t = torch.randn(2, N_BINS, 16).to(device)
        d0 = torch.zeros(2, dtype=torch.long, device=device)
        e = (model.decoder.film(t, d0) - t).abs().max().item()
    print(f'单库 FiLM 恒等性自检: max|film(x)-x| = {e:.2e}  （应为 0）')


# ------------------------ mean-HRTF 基线（分库） ------------------------
# 并库之后单一基线没意义：各库电平约定不同，池化后的均值对谁都不是好基线。
TAGS = [d['tag'] for d in DATASETS]
baselines = {}
with torch.no_grad():
    for d, tr, va in zip(DATASETS, tr_parts, va_parts):
        if len(va) == 0: continue
        mu = torch.stack([tr[i][0] for i in range(len(tr))]).mean(0, keepdim=True)
        V = torch.stack([va[i][0] for i in range(len(va))])
        baselines[d['tag']] = torch.sqrt(((V - mu)[:, band, :] ** 2).mean()).item()
        del V
print(f"\nmean-HRTF 基线 ({config['band_lo_hz']}-{config['band_hi_hz']} Hz, 各库自己的均值):")
for k, v in baselines.items():
    print(f'    {k:<9}{v:.3f} dB')


def evaluate(loader):
    """返回 (总体 RMSE, 总体 MAE, {库: RMSE})"""
    model.eval()
    se = torch.zeros(len(TAGS), device=device); ae = torch.zeros(len(TAGS), device=device)
    cnt = torch.zeros(len(TAGS), device=device)
    with torch.no_grad():
        for x, did, _ in loader:
            x, did = x.to(device), did.to(device)
            rec, _ = model(x, did)
            dd = (rec - x)[:, band_dev, :]
            npt = dd[0].numel()
            for k in range(len(TAGS)):
                m = did == k
                if m.any():
                    se[k] += (dd[m] ** 2).sum(); ae[k] += dd[m].abs().sum()
                    cnt[k] += m.sum() * npt
    per = {t: (se[k] / cnt[k]).sqrt().item() if cnt[k] > 0 else float('nan')
           for k, t in enumerate(TAGS)}
    return (se.sum() / cnt.sum()).sqrt().item(), (ae.sum() / cnt.sum()).item(), per

In [ ]:
# ==========================================================================
# Cell 4 — 训练
#
# resume 只用于【同一次实验被 Colab 断线】的续训。
# 改过 DATASETS（加库）之后不要 resume —— 网络结构变了，而且 warm-start 的
# 结果和单库阶段不可比。加库就把 resume 关掉，从零重训。
# ==========================================================================
history = {'train_rmse': [], 'val_rmse': [], 'val_mae': [], 'train_loss_total': [],
           'per_dataset': []}
best_val_rmse, done_epochs = float('inf'), 0

if resume:
    if not os.path.exists(latest_path):
        raise FileNotFoundError(f'开了 resume 但没有存档: {latest_path}')
    ck = torch.load(latest_path, map_location=device, weights_only=False)
    n_ds_ck, n_bins_ck = ck['model_state_dict']['decoder.film.beta'].shape
    assert n_bins_ck == config['num_bins'], \
        f"存档 {n_bins_ck} bin vs 当前 {config['num_bins']} bin —— freq_cap_hz 改过了"
    assert n_ds_ck == config['num_datasets'], (
        f"存档 {n_ds_ck} 个库 vs 当前 {config['num_datasets']} 个 —— "
        '加库之后请把 resume 关掉，从零重训')
    model.load_state_dict(ck['model_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    history = ck.get('history', history)
    best_val_rmse = ck.get('val_rmse', float('inf'))
    done_epochs = ck.get('epoch', -1) + 1
    for pg in optimizer.param_groups:
        pg['lr'] = config['lr']                  # 热重启
    epochs = additional_epochs
    print(f'从第 {done_epochs} 轮恢复，历史最佳 {best_val_rmse:.4f}，本次再跑 {epochs} 轮')
else:
    epochs = fresh_epochs
    print(f'从零开始训练 {epochs} 轮')

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

for ep in range(epochs):
    model.train()
    tot, mse_sum, nb = 0.0, 0.0, 0
    for x, did, _ in train_loader:
        x, did = x.to(device), did.to(device)
        optimizer.zero_grad()
        rec, _ = model(x, did)
        loss_mse = criterion_mse(rec[:, band_dev, :], x[:, band_dev, :])
        loss_spe = criterion_spectral(rec, x)
        loss = config['w_mse'] * loss_mse + config['w_spectral'] * loss_spe
        loss.backward(); optimizer.step()
        tot += loss.item(); mse_sum += loss_mse.item(); nb += 1
    scheduler.step()

    va_rmse, va_mae, per = evaluate(val_loader)
    tr_rmse = (mse_sum / nb) ** .5
    history['train_loss_total'].append(tot / nb)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(va_rmse)
    history['val_mae'].append(va_mae)
    history['per_dataset'].append(per)

    extra = ('' if len(TAGS) == 1 else
             ' | ' + '  '.join(f'{t} {per[t]:.3f}' for t in TAGS))
    gap = baselines[TAGS[0]] - va_rmse if len(TAGS) == 1 else None
    print(f'Epoch {done_epochs+ep+1:>3} | train {tr_rmse:.4f} | val {va_rmse:.4f} dB '
          f'| MAE {va_mae:.4f}' + (f' | vs baseline {gap:+.3f} dB' if gap is not None else '')
          + extra + f' | lr {scheduler.get_last_lr()[0]:.2e}')

    ck = {'epoch': done_epochs + ep, 'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(), 'history': history,
          'val_rmse': va_rmse, 'config': config, 'split': SPLIT,
          'datasets': [{k: v for k, v in d.items() if k != 'zips'} for d in DATASETS]}
    torch.save(ck, latest_path)
    if va_rmse < best_val_rmse:
        best_val_rmse = va_rmse
        torch.save(ck, best_path)
        print(f'    ↳ 新最佳，已写入 {os.path.basename(best_path)}')

print(f'\n训练结束。最佳 val RMSE = {best_val_rmse:.4f} dB')
for k, v in baselines.items():
    print(f'    {k} 的 mean-HRTF 基线 {v:.3f} dB')

In [ ]:
# ==========================================================================
# Cell 5 — 载入 checkpoint（只做评估时从这里开始，跳过 Cell 4）
# ==========================================================================
ck = torch.load(best_path, map_location=device, weights_only=False)
n_ds_ck, n_bins_ck = ck['model_state_dict']['decoder.film.beta'].shape
assert (n_bins_ck, n_ds_ck) == (config['num_bins'], config['num_datasets']), (
    f"存档 {n_ds_ck} 库 / {n_bins_ck} bin，当前 {config['num_datasets']} 库 / "
    f"{config['num_bins']} bin —— 配置对不上")
model.load_state_dict(ck['model_state_dict'])
model.eval()
history = ck.get('history', {})
print(f"载入 {os.path.basename(best_path)} | epoch {ck['epoch']} | "
      f"val RMSE {ck['val_rmse']:.4f} dB")

# 用存档里的划分，确保评估用的是同一批人
if 'split' in ck:
    SPLIT = ck['split']
    va_parts = [MonauralHRTFDataset(d, SPLIT[d['tag']]['val'],  MIRROR) for d in DATASETS]
    te_parts = [MonauralHRTFDataset(d, SPLIT[d['tag']]['test'], MIRROR) for d in DATASETS]
    val_loader  = DataLoader(ConcatDataset(va_parts), batch_size=config['batch_size'], shuffle=False)
    test_loader = DataLoader(ConcatDataset(te_parts), batch_size=config['batch_size'], shuffle=False)
    for d in DATASETS:
        s = SPLIT[d['tag']]
        print(f"  {d['tag']:<9} train {len(s['train']):>4} / val {len(s['val']):>3} / "
              f"test {len(s['test']):>3} 人")

# 多库时：把各库学到的 EQ 曲线画出来
if config['num_datasets'] > 1:
    import matplotlib.pyplot as plt
    with torch.no_grad():
        lg = (model.decoder.film.log_gamma
              - model.decoder.film.log_gamma.mean(0, keepdim=True)).cpu().numpy()
        bt = (model.decoder.film.beta
              - model.decoder.film.beta.mean(0, keepdim=True)).cpu().numpy()
    fig, ax = plt.subplots(1, 2, figsize=(13, 4), dpi=120)
    for k, t in enumerate(TAGS):
        ax[0].plot(freqs / 1000, 20 * np.log10(np.exp(lg[k])), lw=2, label=t)
        ax[1].plot(freqs / 1000, bt[k], lw=2, label=t)
    ax[0].set_title('FiLM gain 20log10(γ)  [dB]'); ax[1].set_title('FiLM offset β  [dB]')
    for a in ax:
        a.set_xlabel('frequency (kHz)'); a.axhline(0, color='gray', lw=.8)
        a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print('这两条曲线就是各库的"测量染色"。应该看起来像 EQ —— 平缓、宽包络。'
          '如果出现 notch 形状的结构，说明 FiLM 在吸收本该留给 latent 的人群解剖差异。')

In [ ]:
# ==========================================================================
# Cell 6 — 训练曲线
# ==========================================================================
import matplotlib.pyplot as plt

ep = np.arange(1, len(history['val_rmse']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), dpi=120)

ax[0].plot(ep, history['train_rmse'], label='train RMSE', lw=2)
ax[0].plot(ep, history['val_rmse'], label='val RMSE', lw=2)
if len(TAGS) > 1 and history.get('per_dataset'):
    for t in TAGS:
        ax[0].plot(ep, [h.get(t, np.nan) for h in history['per_dataset']],
                   lw=1.2, ls=':', label=f'val {t}')
for t, v in baselines.items():
    ax[0].axhline(v, color='gray', ls='--', lw=1, alpha=.6,
                  label=f'{t} mean-HRTF ({v:.2f} dB)')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('RMSE (dB)')
ax[0].set_title(f"{config['model_type']}  ({config['band_lo_hz']}-{config['band_hi_hz']} Hz)")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].semilogy(ep, history['train_loss_total'], lw=2, label='train total loss')
ax[1].plot(ep, history['val_mae'], lw=2, label='val MAE (dB)')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title('loss / MAE')

plt.tight_layout(); plt.show()
best_ep = int(np.argmin(history['val_rmse'])) + 1
print(f"最佳 epoch {best_ep}: val RMSE {min(history['val_rmse']):.4f} dB")

In [ ]:
# ==========================================================================
# Cell 7 — 频谱重建对比（5 个代表方向，左右耳）
# ==========================================================================
import matplotlib.pyplot as plt
import healpy as hp

DB_PICK = 0                                   # 多库时想看别的库改这里
part = va_parts[DB_PICK]
DID = DATASETS[DB_PICK]['id']
val_subjects = sorted({it[0] for it in part.items})
TARGET = val_subjects[0]
print(f"{DATASETS[DB_PICK]['tag']} 受试者 {TARGET}   "
      f"(val 共 {len(val_subjects)} 人: {val_subjects[:6]} ...)")


def predict_ear(sid, side):
    """返回 (gt, pred)，都已还原到【该耳自己的原始坐标系】。
    模型工作在左耳坐标系；右耳输入被镜像过，输出要再镜像一次还原（对合）。"""
    for s, sd_, path in part.items:
        if s == sid and sd_ == side:
            x = np.load(path).astype(np.float32)[:N_BINS]
            if FLAT:
                x[:FLAT] = x[FLAT]
            xin = x[:, MIRROR] if side == 'right' else x
            with torch.no_grad():
                p, _ = model(torch.from_numpy(xin)[None].to(device),
                             torch.full((1,), DID, dtype=torch.long, device=device))
            p = p[0].cpu().numpy()
            if side == 'right':
                p = p[:, MIRROR]
            return x, p
    return None, None


gt_l, pred_l = predict_ear(TARGET, 'left')
gt_r, pred_r = predict_ear(TARGET, 'right')

directions = {
    'Front (az 0, el 0)':          (np.pi / 2, 0.0),
    'Left (az 90, el 0)':          (np.pi / 2, np.pi / 2),
    'Right (az -90, el 0)':        (np.pi / 2, -np.pi / 2),
    'Left upper (az 45, el 45)':   (np.pi / 4, np.pi / 4),
    'Right upper (az -45, el 45)': (np.pi / 4, -np.pi / 4),
}
freqs_khz = freqs / 1000.0

fig, axes = plt.subplots(len(directions), 1, figsize=(13, 4 * len(directions)), dpi=120)
for ax, (name, (th, ph)) in zip(axes, directions.items()):
    pix = hp.ang2pix(config['nside'], th, ph)
    ax.plot(freqs_khz, gt_l[:, pix],   color='#1f77b4', lw=2.2, label='Left  GT')
    ax.plot(freqs_khz, pred_l[:, pix], color='#00BFFF', lw=2.0, ls='--', label='Left  Pred')
    if gt_r is not None:
        ax.plot(freqs_khz, gt_r[:, pix],   color='#d62728', lw=2.2, label='Right GT')
        ax.plot(freqs_khz, pred_r[:, pix], color='#ff7f0e', lw=2.0, ls='--', label='Right Pred')
    ax.axvspan(6, 12, color='orange', alpha=.10)
    ax.axvspan(0, config['band_lo_hz'] / 1000, color='gray', alpha=.12)
    ax.set_xlim(0, freqs_khz[-1]); ax.set_xlabel('frequency (kHz)')
    ax.set_ylabel('magnitude (dB)'); ax.set_title(f'{TARGET} — {name}')
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"横轴在 {config['band_hi_hz']} Hz 截断；灰色 = <500 Hz 不计损失；"
      '橙色 = 6-12 kHz notch 带')

In [ ]:
# ==========================================================================
# Cell 8 — HEALPix 空间热力图（GT / Pred / 误差）
# ==========================================================================
import matplotlib.pyplot as plt
import healpy as hp


def plot_healpix_map(gt, pred, target_hz, subject, ear='Left'):
    # ★ v1 这里用 16000/128 算频率索引是错的：bin 步进应为 fs/256。
    df = config['sample_rate'] / 256.0
    k = int(round(target_hz / df))
    k = min(max(k, 0), config['num_bins'] - 1)
    f_actual = k * df

    g, p = gt[k, :], pred[k, :]
    err = np.abs(g - p)
    vmin, vmax = min(g.min(), p.min()), max(g.max(), p.max())

    fig = plt.figure(figsize=(18, 5.5), dpi=120)
    plt.suptitle(f'{subject} — {ear} ear @ {f_actual:.0f} Hz    MAE {err.mean():.2f} dB',
                 fontsize=15, fontweight='bold', y=1.02)
    hp.mollview(g,   fig=fig.number, sub=(1, 3, 1), title='Ground truth',
                min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.1f')
    hp.mollview(p,   fig=fig.number, sub=(1, 3, 2), title='Prediction',
                min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.1f')
    hp.mollview(err, fig=fig.number, sub=(1, 3, 3), title='|error|',
                cmap='hot', unit='dB', format='%.1f')
    plt.tight_layout(); plt.show()


for hz in (1000, 8000, 12000):      # 低频 / notch 带 / 高频
    plot_healpix_map(gt_l, pred_l, hz, TARGET, 'Left')

# ---- 逐频率误差曲线：看清误差集中在哪个频段 ----
err_f = np.sqrt(((gt_l - pred_l) ** 2).mean(axis=1))
plt.figure(figsize=(11, 4), dpi=120)
plt.plot(freqs / 1000, err_f, lw=2)
plt.axvspan(6, 12, color='orange', alpha=.12, label='notch band')
plt.axvspan(0, config['band_lo_hz'] / 1000, color='gray', alpha=.12, label='out of loss band')
plt.xlabel('frequency (kHz)'); plt.ylabel('RMS error (dB)')
plt.title(f'{TARGET} left ear — error vs frequency')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# ==========================================================================
# Cell 9 — ★ TEST 集评估（整个项目里只在最后跑一次）
#
# 规矩：调超参、选 epoch、比架构，全部看 val。test 只在结论定稿时跑一次，
#      跑完不要再回去改任何东西 —— 否则它就退化成第二个 val 集。
# ==========================================================================
RUN_TEST = False        # ← 定稿时改成 True

if not RUN_TEST:
    print('test 集未评估。确认所有决定都已基于 val 做完之后，把 RUN_TEST 改为 True。')
else:
    te_rmse, te_mae, te_per = evaluate(test_loader)
    va_rmse, va_mae, va_per = evaluate(val_loader)

    # test 的 mean-HRTF 基线用 train 的均值算，不能用 test 自己的
    te_base = {}
    with torch.no_grad():
        for d, tr, te in zip(DATASETS, tr_parts, te_parts):
            if len(te) == 0: continue
            mu = torch.stack([tr[i][0] for i in range(len(tr))]).mean(0, keepdim=True)
            T = torch.stack([te[i][0] for i in range(len(te))])
            te_base[d['tag']] = torch.sqrt(((T - mu)[:, band, :] ** 2).mean()).item()
            del T

    print(f"{'库':<10}{'test人数':>9}{'val RMSE':>11}{'test RMSE':>11}"
          f"{'test基线':>10}{'相对基线':>10}")
    for d in DATASETS:
        t = d['tag']; b = te_base.get(t, float('nan'))
        print(f"{t:<10}{len(SPLIT[t]['test']):>9}{va_per.get(t, float('nan')):>11.3f}"
              f"{te_per.get(t, float('nan')):>11.3f}{b:>10.3f}"
              f"{b - te_per.get(t, float('nan')):>+10.3f}")
    if len(TAGS) > 1:
        print(f"{'pooled':<10}{'':>9}{va_rmse:>11.3f}{te_rmse:>11.3f}")
    print(f'\ntest MAE {te_mae:.4f} dB')
    print('\n如果 test 明显差于 val，说明选 epoch / 调参时把 val 用过头了。')

## 验收测试

1. **镜像是否生效** —— 把 `analysis/latent_variance_decomp.py` 在新 checkpoint 上再跑一遍：

| | v1（未镜像） | v2 期望 |
|---|---|---|
| `'哪只耳'` 方差占比 | 71.7% | **接近 0** |
| `'哪个人'` 方差占比 | 18.6% | **大幅上升** |

2. **FiLM 是否越界**（多库时）—— Cell 5 会画出各库的 γ / β 曲线。它们应该看起来像
   **EQ**：平缓、宽包络。如果出现 notch 形状的结构，说明 FiLM 在吸收本该留给
   latent 的人群解剖差异，那正是 Craig 否定过的方向。

3. **并库到底有没有收益** —— 两次都从零训练（本 notebook 的默认路径就是从零），
   同 seed 同轮数，在**同一个 CARLab test 集**上比。因为 `split_v21.json` 里
   CARLab 的划分在加库前后完全不变（每个库用自己的 rng），这个对比是干净的。